# Module 05 Essentials – Extended Text Classification

This notebook is a continuation of the work completed in **Module 05 Basics**, where I built a Naive Bayes classifier to predict whether Jeopardy! questions are “high value” or “low value” based on their textual features.

In this extended analysis, I build on the same cleaned dataset and preprocessing steps, but add **two additional classification approaches**:

1. **Linear Support Vector Machine (SVM)** using TF–IDF features  
2. **A simple TensorFlow neural network classifier**

These models are evaluated alongside the baseline Naive Bayes model to compare performance across different machine learning approaches.

The new analysis begins at **Section 6)** below.


In this module, I built a text classification pipeline using a Naive Bayes classifier to predict whether Jeopardy! questions are “high value” or “low value” based on their wording. After loading and cleaning the dataset, I tokenized the text, vectorized it, and trained a baseline model to see how well linguistic features can predict difficulty.

#1) Set up and import.  Loading the environment:

In [2]:
from google.colab import files
uploaded = files.upload()  #uploading from local computer

Saving jeopardy.json to jeopardy.json


In [3]:
import os, random
import numpy as np
import json
import pandas as pd

SEED = 123  # set.seed in R
random.seed(SEED); np.random.seed(SEED)

# Get the uploaded filename (colab sets automatically)
local_name = list(uploaded.keys())[0]

JSON_PATH = f"/content/{local_name}" #construct full path

with open(JSON_PATH, "r", encoding="utf-8") as f: #load json in python
    data = json.load(f)

df = pd.DataFrame(data) #convert json to panas
print("Records:", len(df))
df.head() #sanity checks


Records: 216930


,category,air_date,question,value,answer,round,show_number
0,HISTORY,2004-12-31,"'For the last 8 years of his life, Galileo was...",$200,Copernicus,Jeopardy!,4680
1,ESPN's TOP 10 ALL-TIME ATHLETES,2004-12-31,'No. 2: 1912 Olympian; football star at Carlis...,$200,Jim Thorpe,Jeopardy!,4680
2,EVERYBODY TALKS ABOUT IT...,2004-12-31,'The city of Yuma in this state has a record a...,$200,Arizona,Jeopardy!,4680
3,THE COMPANY LINE,2004-12-31,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,McDonald\'s,Jeopardy!,4680
4,EPITAPHS & TRIBUTES,2004-12-31,"'Signer of the Dec. of Indep., framer of the C...",$200,John Adams,Jeopardy!,4680


2)  Clean value column.  Assign "low" = 0 or "high"  = 1 to value.  $1000 is used as the cutoff, and is the cutoff between first and second round questions.

In [5]:
import re

def clean_value(v):
    # handle missing or non-string values
    if v is None:
        return 0
    if not isinstance(v, str):
        v = str(v)

    v = v.strip().lower()

    # common placeholders in dataset, debugging
    if v in {"", "none", "null"}:
        return 0

    # keep only digits
    digits = re.sub(r"[^\d]", "", v)
    return int(digits) if digits else 0

df["value_clean"] = df["value"].apply(clean_value)
df["high_value"] = (df["value_clean"] >= 1000).astype(int) #assigns 0=low, 1=high values

Check, debugging:

In [6]:
df["value_clean"].max()

18000

In [7]:
print(df["value"].isna().sum(), "rows had value = None")
print(df["value_clean"].describe())

# make sure both classes are present:
df["high_value"].value_counts()


3634 rows had value = None
count    216930.000000
mean        739.988476
std         639.822693
min           0.000000
25%         400.000000
50%         600.000000
75%        1000.000000
max       18000.000000
Name: value_clean, dtype: float64


,count
high_value,
0,155622
1,61308


In [8]:
df[["value", "value_clean", "high_value"]].head()

,value,value_clean,high_value
0,$200,200,0
1,$200,200,0
2,$200,200,0
3,$200,200,0
4,$200,200,0


#3)  Build and test tokenizer:

In [9]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re

# tokenization models (for word_tokenize)
nltk.download("punkt_tab", quiet=True) #gemini debug
nltk.download("stopwords", quiet=True)

STOPWORDS = set(stopwords.words("english")) #set of english stopwords

def simple_tokenizer(text):  #using docstring for function
    """
    Simple tokenizer for Jeopardy questions:
      - lowercase
      - remove punctuation/digits
      - tokenize
      - remove stopwords and 1-letter tokens
    """
    if not isinstance(text, str):
        return []

    text = text.lower() #lowercase
    text = re.sub(r"[^a-z\s]", " ", text) #remove everything except letters/spaces
    tokens = word_tokenize(text) #tokenize
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return tokens


Sanity check:

In [11]:
print(simple_tokenizer(df.loc[0, "question"]))


['last', 'years', 'life', 'galileo', 'house', 'arrest', 'espousing', 'man', 'theory']


#4) Vectorize the questions

Count Vectorizer transforms text to matrix, where each word is a numeric feature.

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer=simple_tokenizer, min_df=2) #apply cleaned tokenizer, removes rare words
X = vectorizer.fit_transform(df["question"]) #matrix of numeric values
y = df["high_value"] # binary column: low/high value labels


sanity check

In [13]:
print("X shape:", X.shape) #predictors
print("y shape:", y.shape) #outcome

X shape: (216930, 49019)
y shape: (216930,)


In [14]:
df[["question", "high_value"]].head()

,question,high_value
0,"'For the last 8 years of his life, Galileo was...",0
1,'No. 2: 1912 Olympian; football star at Carlis...,0
2,'The city of Yuma in this state has a record a...,0
3,"'In 1963, live on ""The Art Linkletter Show"", t...",0
4,"'Signer of the Dec. of Indep., framer of the C...",0


In [15]:
question = df["question"].iloc[0] #look at tokens first question
tokens = simple_tokenizer(question)
print("Original:", question)
print("Tokens:", tokens)

Original: 'For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory'
Tokens: ['last', 'years', 'life', 'galileo', 'house', 'arrest', 'espousing', 'man', 'theory']


#5) Train/test split and NB Model:

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

Xtr, Xte, ytr, yte = train_test_split(
    X, y,
    test_size=0.25, #75/25 split
    random_state=SEED, #set seed
    stratify=y  #strata in tidymodels
)

nb = MultinomialNB() #define model, engine...
nb.fit(Xtr, ytr)  #train on training data

preds = nb.predict(Xte)  #produce predicted classes

nb_acc = accuracy_score(yte, preds)
print("Naive Bayes accuracy:", round(nb_acc, 3)) #requested for assignment


Naive Bayes accuracy: 0.691


The Naive Bayes classifier provides a baseline for this task.  Using a train/test split, the model achieved an accuracy of 0.691, which was expected given the limited cues available for predicting difficulty. Although the performance is modest, the exercise establishes a baseline and sets the stage for more advanced methods in the Module 05 Essentials extension.

#6)  module-05-essentials extension

Continuing from **Module 05 Basics** (where I built a Naive Bayes baseline classifier), this section extends the analysis by training two additional text classification models:

1. **Linear SVM (TF–IDF)**
2. **TensorFlow Neural Network (using integer-sequence embeddings)**

After training these models, I compare their accuracies against the Naive Bayes baseline to evaluate performance differences across model types.



### 7) Linear SVM Classifier (TF–IDF Features)

To extend the Naive Bayes baseline from Module 05 Basics, I first train a **Linear Support Vector Machine (SVM)** using TF–IDF vectorized text.  
SVMs are margin-based classifiers that try to find the best linear boundary separating the two classes (“high” vs. “low" value).  
TF–IDF helps weight informative terms more strongly while down-weighting common words.

This section vectorizes the Jeopardy! questions using the same custom tokenizer as before, trains a LinearSVC model, and evaluates its accuracy on the test set.


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1) Vectorize text with TFIDF
tfidf = TfidfVectorizer(
    analyzer=simple_tokenizer,  # uses function from section #3)
    min_df=2,                   # drop rare terms
    max_df=0.95                 # drop extremely common terms
)

X_tfidf = tfidf.fit_transform(df["question"]) #fit model and transform

y = df["high_value"].astype(int).values  # same 0/1 int target labels

# 2) Train/test split
Xtr, Xte, ytr, yte = train_test_split(
    X_tfidf, y,
    test_size=0.25,
    random_state=SEED, #set.seed
    stratify=y  #keeps class balance
)

# 3) Train and evaluate
svm = LinearSVC()
svm.fit(Xtr, ytr)
svm_preds = svm.predict(Xte) #predict on test set

svm_acc = accuracy_score(yte, svm_preds) #compute accuracy
print("Linear SVM (TF-IDF) accuracy:", round(svm_acc, 3))

Linear SVM (TF-IDF) accuracy: 0.69


### SVM Results

The Linear SVM achieved an accuracy of **0.690**, which is close to the Naive Bayes baseline.  
This is expected because Jeopardy! clues are short, sparse, and contain limited linguistic signal separating “easy” from “hard.”  
SVMs often perform very well on text, but in this dataset the signal is subtle, so the improvement over NB is questionable.


### 8) TensorFlow Model (Neural Network Classifier)

For my second extended model, I used a small TensorFlow/Keras neural network. I’ve used TensorFlow/Keras once before in my BDS II class, but this is my first time applying it to text classification in Python.

This model works differently from Naive Bayes and SVM because it learns its own internal word representations instead of relying on TF-IDF features. I split the data before vectorizing, adapted the vocabulary on the training text, and then trained a simple model with an embedding layer, a pooling layer, and a sigmoid classifier.



In [34]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Label prep (0/1 ints) as above
yy = df["high_value"].astype(int).values

# 1) Train/test split
Xtr_text, Xte_text, ytr_tf, yte_tf = train_test_split(
    df["question"].values, yy,
    test_size=0.25,
    random_state=SEED,
    stratify=yy
)

# 2) Text vectorizer
VOCAB = 15000 #limit to most frequent words
SEQ_LEN = 30 #truncate to 30 tokens

text_vec = layers.TextVectorization(
    max_tokens=VOCAB,  #most common words kept
    output_mode="int",  #converts to integer
    output_sequence_length=SEQ_LEN,  #forces same length questions
    standardize="lower_and_strip_punctuation", #basic cleaning
    split="whitespace" #splits words by spaces
)

text_vec.adapt(Xtr_text)  # call/adapt on train

# 3) Convert text to integer sequences
Xtr_int = text_vec(Xtr_text)
Xte_int = text_vec(Xte_text)

# 4) Build a simple NN model
model = tf.keras.Sequential([
    layers.Embedding(input_dim=VOCAB, output_dim=64), #learns word distributions
    layers.GlobalAveragePooling1D(), #combines into one vector
    layers.Dense(32, activation="relu"), #high vs low value
    layers.Dense(1, activation="sigmoid") #probability output
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy", #std for binary
    metrics=["accuracy"])

# 5) Train model
H = model.fit(
    Xtr_int, ytr_tf,
    epochs=5, batch_size=64,
    validation_split=0.1,
    verbose=1
)

# 6) Evaluate
loss, acc = model.evaluate(Xte_int, yte_tf, verbose=0)
tf_acc = float(acc)
print("TensorFlow model accuracy:", round(tf_acc, 3))

Epoch 1/5
2288/2288 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - accuracy: 0.7186 - loss: 0.5918 - val_accuracy: 0.7201 - val_loss: 0.5799
Epoch 2/5
2288/2288 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - accuracy: 0.7228 - loss: 0.5665 - val_accuracy: 0.7204 - val_loss: 0.5817
Epoch 3/5
2288/2288 ━━━━━━━━━━━━━━━━━━━━ 32s 14ms/step - accuracy: 0.7289 - loss: 0.5508 - val_accuracy: 0.7116 - val_loss: 0.5894
Epoch 4/5
2288/2288 ━━━━━━━━━━━━━━━━━━━━ 27s 12ms/step - accuracy: 0.7376 - loss: 0.5366 - val_accuracy: 0.7088 - val_loss: 0.5932
Epoch 5/5
2288/2288 ━━━━━━━━━━━━━━━━━━━━ 28s 12ms/step - accuracy: 0.7456 - loss: 0.5232 - val_accuracy: 0.7033 - val_loss: 0.6070
TensorFlow model accuracy: 0.702


The TensorFlow model trained smoothly and reached an accuracy of 0.702.
Even though it uses a simple architecture, the embedding layer helped it capture useful word patterns in the Jeopardy! questions. The model was stable, did not overfit with five epochs, and produced consistent results across runs using the same random seed.

#9) Model comparison and conclusion:

In [36]:
results = pd.DataFrame({
    "Model": ["Naive Bayes (TF-IDF)", "Linear SVM (TF-IDF)", "TensorFlow (NN)"],
    "Accuracy": [nb_acc, svm_acc, tf_acc]
})
results.style.format({"Accuracy": "{:.3f}"})

,Model,Accuracy
0,Naive Bayes (TF-IDF),0.691
1,Linear SVM (TF-IDF),0.690
2,TensorFlow (NN),0.702


We compared three text classifiers on the Jeopardy! high/low-value task. The TensorFlow model achieved the highest accuracy (0.702), with Naive Bayes (0.691) and Linear SVM (0.690) close behind. The ~1–2 percentage-point spread is modest and may not be statistically significant on a single random split. Given the brevity and overlap of Jeopardy! questions, this is a challenging problem for simple models. Future gains will likely come from feature engineering (e.g., character/word n-grams, tuned min_df, lemmatization), hyperparameter tuning (e.g., SVM C, class weights), and more robust evaluation via cross-validation.

It is not surprising that the SVM model did not outperform Naive Bayes. Jeopardy clues are short, sparse, and highly formulaic, so a simple bag-of-words model like NB can perform nearly as well as a margin-based classifier. The distinction between ‘high value’ and ‘low value’ clues contains limited linguistic signal, since difficulty is often based on background knowledge rather than wording. As a result, both NB and SVM cluster around similar accuracy values, with the TensorFlow model showing only a modest improvement.

The TensorFlow model showed a modest improvement (≈0.702) over Naive Bayes (0.691) and SVM (0.690). This result is expected because neural networks can model nonlinear interactions in the TF-IDF space through hidden layers and activation functions. They can learn combinations of word features and reduce noise more flexibly than linear models. However, the improvement remains small because Jeopardy clue difficulty contains very weak linguistic signal—many ‘easy’ and ‘hard’ clues look nearly identical in surface form.

Although the TensorFlow model achieved a slightly higher accuracy (0.702) than the Naive Bayes (0.691) and SVM (0.690) models, this small difference is not statistically significant, as it falls within expected random variation for a dataset of this size. However, the neural model’s modest edge aligns with its greater representational capacity—it can capture limited aspects of word order and semantic context that simpler bag-of-words models overlook. With a larger or more complex corpus, this performance gap would possibly widen.